# Module 5: The Full Walkthrough — Building RAG from Scratch

You've learned the concepts. Now you'll build the whole pipeline in one sitting: ingest a document, watch a plain prompt fail, fix it with retrieval, and ship the result as an API.

The document we're ingesting is the final scene of *Hamlet* — with every character renamed. The renaming matters: the model has read *Hamlet* a thousand times in training, but it has never met **Queen Maribel**. If it answers questions about her correctly, that answer came from *your* document, not from memorized training data. That's the proof that retrieval is working.

**What you'll build:**
1. A document store (one renamed Shakespeare scene, chunked)
2. A baseline prompt — no context — that fails
3. Embedding-based retrieval with `sentence-transformers`
4. A RAG prompt that succeeds
5. A working REST API around the whole pipeline

**Requirements:** a free Groq API key from [console.groq.com](https://console.groq.com), Python 3.10+.

## Step 0 — Setup

Install the three libraries we need. `groq` talks to the LLM, `sentence-transformers` creates embeddings locally (no API key needed for this part), and `fastapi`/`uvicorn` power the API at the end.

In [ ]:
%pip install -q groq sentence-transformers fastapi uvicorn requests

In [ ]:
import os

# Set your Groq API key here or export GROQ_API_KEY before launching Jupyter
# os.environ["GROQ_API_KEY"] = "gsk_your_key_here"

from groq import Groq

client = Groq()  # reads GROQ_API_KEY from the environment
MODEL = "llama-3.3-70b-versatile"

def ask_llm(prompt: str, system: str = "You are a helpful assistant.") -> str:
    """One call to the LLM. Every example in this notebook goes through here."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content

print("Client ready.")

## Step 1 — The Document

This is the last scene of *Hamlet*, condensed into plain prose, with the cast renamed:

| Original | Renamed |
|---|---|
| Hamlet | Prince Corvin |
| Claudius | King Aldous |
| Gertrude | Queen Maribel |
| Laertes | Lord Fenwick |
| Horatio | Tobias |

Each paragraph below becomes one **chunk** in our document store. In a real system your chunker would split documents automatically (you saw strategies for this in Module 3); here we chunk by hand so you can see exactly what the retriever is choosing between.

In [ ]:
DOCUMENT_CHUNKS = [
    # 1
    """At the royal castle of Elsinar, Prince Corvin told his friend Tobias how he had
discovered King Aldous's sealed letter ordering Corvin's execution in England, and how
he had rewritten the letter so that the two courtiers carrying it were executed instead.""",

    # 2
    """A foppish courtier named Ostrand arrived with a challenge: King Aldous had wagered
six fine horses that Prince Corvin could not best Lord Fenwick in a fencing match of
twelve passes. Corvin accepted the challenge, though a strange misgiving stirred in
his heart, which he confessed to Tobias.""",

    # 3
    """What Prince Corvin did not know was that King Aldous and Lord Fenwick had laid a
double trap. Fenwick's rapier would be unbated — sharp, not blunted — and its tip
anointed with a poison so deadly that a scratch would kill. As a backup, King Aldous
prepared a poisoned cup of wine to offer Corvin between passes.""",

    # 4
    """Before the match began, Prince Corvin offered Lord Fenwick a gracious apology for
the wrongs he had done him, blaming his madness rather than his will. Fenwick said he
was satisfied in nature, though his honor still demanded the match proceed.""",

    # 5
    """The court assembled. Corvin won the first pass and the second. King Aldous dropped
a poisoned pearl into a cup of wine and offered it to Corvin, saying he drank to the
prince's health. Corvin waved the cup away, wanting to finish the bout first.""",

    # 6
    """Queen Maribel, delighted by her son's success, took up the poisoned cup herself to
toast him. King Aldous cried out for her not to drink, but too late — Queen Maribel
drank the poisoned wine that had been meant for Prince Corvin.""",

    # 7
    """In the scuffle that followed, Lord Fenwick wounded Prince Corvin with the poisoned
blade; then, in a violent exchange, the two rapiers were switched, and Corvin wounded
Fenwick with the same envenomed point.""",

    # 8
    """Queen Maribel collapsed, crying out that the drink was poisoned, and died. The dying
Lord Fenwick, struck by remorse, confessed everything: the blade was unbated and
poisoned, the treachery was the King's, and he himself was justly killed by his own
weapon.""",

    # 9
    """Enraged, Prince Corvin stabbed King Aldous with the poisoned rapier and forced the
remaining poisoned wine down his throat. The King died. Fenwick and Corvin exchanged
forgiveness before Fenwick died.""",

    # 10
    """The poison overcame Prince Corvin. He begged Tobias not to follow him in death but
to stay alive and tell his story truly. With the words 'the rest is silence,' Corvin
died, and Tobias bade the sweet prince good night as flights of angels sang him to
his rest.""",
]

print(f"{len(DOCUMENT_CHUNKS)} chunks loaded.")

## Step 2 — The Baseline: Ask Without Context

First, the naive approach: just ask the model directly. No document, no retrieval.

The question is one a human who read the scene could answer instantly.

In [ ]:
QUESTION = "How did Queen Maribel die?"

baseline_answer = ask_llm(QUESTION)
print(baseline_answer)

Run it a few times. You'll get some flavor of *"I don't have information about Queen Maribel"* — or worse, a confident hallucination about some other fictional queen. The model isn't broken; it just has no idea who Queen Maribel is. **The knowledge lives in our document, and the model has never seen our document.**

That's the gap RAG closes.

## Step 3 — Embed the Chunks

We turn every chunk into a vector using a local embedding model (`all-MiniLM-L6-v2` — small, fast, free, runs on a laptop). Questions get embedded the same way, and retrieval is just "find the chunk vectors closest to the question vector."

You saw the theory of embeddings in Module 2. This is all the code it takes in practice:

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Embed every chunk once, up front. Real systems store these in a vector DB;
# for 10 chunks, a numpy array IS our vector DB.
chunk_vectors = embedder.encode(DOCUMENT_CHUNKS, normalize_embeddings=True)

print(f"Embedded {chunk_vectors.shape[0]} chunks into "
      f"{chunk_vectors.shape[1]}-dimensional vectors.")

In [ ]:
def retrieve(question: str, top_k: int = 3) -> list[str]:
    """Return the top_k most relevant chunks for a question."""
    q_vec = embedder.encode([question], normalize_embeddings=True)[0]
    # Vectors are normalized, so cosine similarity is just a dot product
    scores = chunk_vectors @ q_vec
    best = np.argsort(scores)[::-1][:top_k]
    return [DOCUMENT_CHUNKS[i] for i in best], [float(scores[i]) for i in best]

chunks, scores = retrieve(QUESTION)
for score, chunk in zip(scores, chunks):
    print(f"[score {score:.3f}] {chunk[:90]}...\n")

Look at what came back: the retriever found the chunk where Maribel drinks the poisoned wine and the chunk where she collapses and dies — without any keyword matching, just semantic similarity. The fencing-wager chunk scores much lower because it's about something else, even though it's from the same scene.

## Step 4 — The RAG Prompt

Now we assemble the pattern you learned in Module 4: retrieved context on top, question on the bottom, and instructions telling the model to answer **only from the context**.

In [ ]:
RAG_SYSTEM = """You are a careful assistant. Answer the user's question using ONLY the
provided context. If the context does not contain the answer, say so plainly.
Do not use any outside knowledge."""

def rag_ask(question: str, top_k: int = 3) -> str:
    chunks, _ = retrieve(question, top_k)
    context = "\n\n---\n\n".join(chunks)
    prompt = f"""CONTEXT:
{context}

QUESTION: {question}"""
    return ask_llm(prompt, system=RAG_SYSTEM)

print(rag_ask(QUESTION))

There it is: *Queen Maribel died by drinking the poisoned wine that King Aldous had prepared for Prince Corvin.* The same model that shrugged in Step 2 now answers correctly — the only thing that changed is what we put in the prompt.

Try a few more to see retrieval steering the answer:

In [ ]:
for q in [
    "How did Lord Fenwick die?",
    "What was the double trap set for Prince Corvin?",
    "What were Prince Corvin's last words?",
    "What was Queen Maribel's favorite food?",   # not in the document!
]:
    print(f"Q: {q}")
    print(f"A: {rag_ask(q)}\n")

The last question is the important one. A well-instructed RAG system doesn't invent an answer — it tells you the context doesn't contain one. That refusal is a feature. It's the difference between a system you can trust and one you have to fact-check.

## Step 5 — Ship It as an API

A notebook proves the concept; an API makes it usable. The cell below writes a complete FastAPI app — the exact same pipeline, wrapped in one `/ask` endpoint. Everything in it is code you've already seen.

In [ ]:
%%writefile rag_api.py
"""A minimal RAG API — the Module 5 pipeline as a web service.

Run with:   uvicorn rag_api:app --reload
Then try:   curl -X POST http://127.0.0.1:8000/ask \
                 -H "Content-Type: application/json" \
                 -d '{"question": "How did Queen Maribel die?"}'
"""
import numpy as np
from fastapi import FastAPI
from pydantic import BaseModel
from groq import Groq
from sentence_transformers import SentenceTransformer

# --- the document (same chunks as the notebook) -------------------------
DOCUMENT_CHUNKS = [
    """At the royal castle of Elsinar, Prince Corvin told his friend Tobias how he had
discovered King Aldous's sealed letter ordering Corvin's execution in England, and how
he had rewritten the letter so that the two courtiers carrying it were executed instead.""",
    """A foppish courtier named Ostrand arrived with a challenge: King Aldous had wagered
six fine horses that Prince Corvin could not best Lord Fenwick in a fencing match of
twelve passes. Corvin accepted the challenge, though a strange misgiving stirred in
his heart, which he confessed to Tobias.""",
    """What Prince Corvin did not know was that King Aldous and Lord Fenwick had laid a
double trap. Fenwick's rapier would be unbated — sharp, not blunted — and its tip
anointed with a poison so deadly that a scratch would kill. As a backup, King Aldous
prepared a poisoned cup of wine to offer Corvin between passes.""",
    """Before the match began, Prince Corvin offered Lord Fenwick a gracious apology for
the wrongs he had done him, blaming his madness rather than his will. Fenwick said he
was satisfied in nature, though his honor still demanded the match proceed.""",
    """The court assembled. Corvin won the first pass and the second. King Aldous dropped
a poisoned pearl into a cup of wine and offered it to Corvin, saying he drank to the
prince's health. Corvin waved the cup away, wanting to finish the bout first.""",
    """Queen Maribel, delighted by her son's success, took up the poisoned cup herself to
toast him. King Aldous cried out for her not to drink, but too late — Queen Maribel
drank the poisoned wine that had been meant for Prince Corvin.""",
    """In the scuffle that followed, Lord Fenwick wounded Prince Corvin with the poisoned
blade; then, in a violent exchange, the two rapiers were switched, and Corvin wounded
Fenwick with the same envenomed point.""",
    """Queen Maribel collapsed, crying out that the drink was poisoned, and died. The dying
Lord Fenwick, struck by remorse, confessed everything: the blade was unbated and
poisoned, the treachery was the King's, and he himself was justly killed by his own
weapon.""",
    """Enraged, Prince Corvin stabbed King Aldous with the poisoned rapier and forced the
remaining poisoned wine down his throat. The King died. Fenwick and Corvin exchanged
forgiveness before Fenwick died.""",
    """The poison overcame Prince Corvin. He begged Tobias not to follow him in death but
to stay alive and tell his story truly. With the words 'the rest is silence,' Corvin
died, and Tobias bade the sweet prince good night as flights of angels sang him to
his rest.""",
]

# --- pipeline setup (runs once, at startup) ------------------------------
MODEL = "llama-3.3-70b-versatile"
RAG_SYSTEM = """You are a careful assistant. Answer the user's question using ONLY the
provided context. If the context does not contain the answer, say so plainly.
Do not use any outside knowledge."""

client = Groq()  # reads GROQ_API_KEY from the environment
embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_vectors = embedder.encode(DOCUMENT_CHUNKS, normalize_embeddings=True)

app = FastAPI(title="Module 5 RAG API")


class AskRequest(BaseModel):
    question: str
    top_k: int = 3


@app.post("/ask")
def ask(req: AskRequest):
    # 1. Retrieve
    q_vec = embedder.encode([req.question], normalize_embeddings=True)[0]
    scores = chunk_vectors @ q_vec
    best = np.argsort(scores)[::-1][: req.top_k]
    chunks = [DOCUMENT_CHUNKS[i] for i in best]

    # 2. Augment
    context = "\n\n---\n\n".join(chunks)
    prompt = f"CONTEXT:\n{context}\n\nQUESTION: {req.question}"

    # 3. Generate
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": RAG_SYSTEM},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
    )
    return {
        "question": req.question,
        "answer": response.choices[0].message.content,
        "sources": [
            {"chunk": DOCUMENT_CHUNKS[i][:80] + "...", "score": float(scores[i])}
            for i in best
        ],
    }

## Step 6 — Run the API Right Here in the Notebook

You could run the server from a terminal (`uvicorn rag_api:app --reload`), but we can also launch it in a background thread and call it over real HTTP without leaving Jupyter. This is the exact same server your users would hit — same port, same JSON, same everything.

In [ ]:
import threading, time
import uvicorn
from rag_api import app   # import the app we just wrote to disk

# Run uvicorn in a background thread so the notebook stays interactive
config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")
server = uvicorn.Server(config)

thread = threading.Thread(target=server.run, daemon=True)
thread.start()

time.sleep(2)  # give the server a moment to start
print("API is running at http://127.0.0.1:8000")

In [ ]:
import requests

response = requests.post(
    "http://127.0.0.1:8000/ask",
    json={"question": "How did Queen Maribel die?"},
)

result = response.json()
print("ANSWER:", result["answer"])
print()
print("SOURCES:")
for src in result["sources"]:
    print(f"  [score {src['score']:.3f}] {src['chunk']}")

That's a genuine HTTP request-response cycle — the same call your web app, Slack bot, or internal tool would make. The response carries the answer **and** the source chunks with similarity scores, the beginnings of citation, which is how production RAG systems earn user trust.

Try a few more questions through the API:

In [ ]:
for q in [
    "Who set the double trap, and what was it?",
    "What happened to King Aldous?",
]:
    r = requests.post("http://127.0.0.1:8000/ask", json={"question": q})
    print(f"Q: {q}")
    print(f"A: {r.json()['answer']}\n")

When you're done, stop the server with the cell below. (If you skip this, the daemon thread simply dies when the notebook kernel shuts down.)

In [ ]:
server.should_exit = True
thread.join(timeout=5)
print("Server stopped.")

**Running it outside the notebook** works exactly as you'd expect — from a terminal:

```bash
uvicorn rag_api:app --reload
```

```bash
curl -X POST http://127.0.0.1:8000/ask \
     -H "Content-Type: application/json" \
     -d '{"question": "How did Queen Maribel die?"}'
```

FastAPI also gives you interactive docs for free at `http://127.0.0.1:8000/docs`, where you can test the endpoint from your browser.

## Make It Yours

Everything above transfers directly to your own data. To adapt it:

1. **Swap the document.** Replace `DOCUMENT_CHUNKS` with your own content — product docs, support articles, policy text. Use a chunker (Module 3) instead of hand-splitting.
2. **Scale the store.** Past a few hundred chunks, move the vectors from a numpy array into a vector database. The retrieve/augment/generate loop doesn't change.
3. **Tune the prompt.** The system prompt is where behavior lives: citation format, tone, refusal rules, answer length. Everything from Module 4 applies here.
4. **Watch the failure modes.** Ask questions your document *can't* answer and make sure the system says so instead of guessing.

That's the whole course in one pipeline: the model provides the language, your documents provide the knowledge, and the prompt is the contract between them.